In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.preprocessing import RobustScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import AdamW

# Load the dataset
file_path = "/content/converted_data_1.csv"
df = pd.read_csv(file_path)

# Filter for E. coli and Meropenem
df = df[(df['Organism'] == 'Escherichia coli') & (df['Antibiotic'] == 'Meropenem')]

# Select the most frequent 'Susceptibility test type'
most_frequent_susceptibility = df['Susceptibility test type'].mode()[0]
df = df[df['Susceptibility test type'] == most_frequent_susceptibility]

# Define columns
date_column = "Date of admission to the hospital"
target_column = "Value"
feature_columns = ['Age', 'Sex', 'District', 'State', 'Country', 'Hospital ID', 'Hospital dept', 'Susceptibility test type']

# Convert date column to datetime
df[date_column] = pd.to_datetime(df[date_column], errors='coerce')

# Convert target column to numeric
df[target_column] = pd.to_numeric(df[target_column], errors='coerce')

# Handle missing values
df['Age'].fillna(df['Age'].median(), inplace=True)
df = df.dropna(subset=[target_column])

# Encode categorical features
encoders = {}
for col in ['Sex', 'District', 'State', 'Country', 'Hospital dept', 'Susceptibility test type']:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = df[col].fillna("Unknown")
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

# Sort by date
df = df.dropna(subset=[date_column]).sort_values(date_column)

# Standardize numerical features using RobustScaler
feature_scaler = RobustScaler()
df[feature_columns] = feature_scaler.fit_transform(df[feature_columns])

# Separate scaler for the target column
target_scaler = RobustScaler()
df[[target_column]] = target_scaler.fit_transform(df[[target_column]])

# Give higher weight to "Susceptibility test type"
df['Susceptibility test type'] *= 5  # Further increasing the impact

# Prepare time-series sequences
def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:i + sequence_length, :-1])
        y.append(data[i + sequence_length, -1])
    return np.array(X), np.array(y)

sequence_length = 15  # Increased sequence length for better pattern recognition
scaled_data = df[feature_columns + [target_column]].values
X, y = create_sequences(scaled_data, sequence_length)

# Split into training and testing sets (80%-20%)
split_idx = int(0.8 * len(X))  # 80% for training, 20% for testing
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Improved LSTM model
model = Sequential([
    LSTM(128, activation='relu', return_sequences=True, input_shape=(sequence_length, X.shape[2])),
    BatchNormalization(),
    Dropout(0.4),
    LSTM(64, activation='relu', return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

# Compile model
optimizer = AdamW(learning_rate=0.0003)
model.compile(optimizer=optimizer, loss='mse')
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Train model
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test),
    epochs=300, batch_size=32, callbacks=[early_stopping], verbose=1
)

# Predictions
predictions = model.predict(X_test)

y_test_scaled = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
pred_scaled = target_scaler.inverse_transform(predictions).flatten()

# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()
 #for a given testset after running
actual_values = np.array([28.00, 26.00, 15.00, 28.00, 30.00, 30.00, 30.00, 24.00]) #y_test_scaled
predicted_values = np.array([26.43, 26.63, 26.81, 27.37, 27.82, 27.78, 27.78, 27.76]) #pred_scaled

# Plot actual vs predicted values for the test dataset
plt.figure(figsize=(12, 6))
sns.lineplot(x=range(len(y_test_scaled)), y=y_test_scaled, label='Actual Test', marker='o')
sns.lineplot(x=range(len(pred_scaled)), y=pred_scaled, label='Predicted Test', marker='x')
plt.xlabel("Sample Index")
plt.ylabel("Meropenem Value")
plt.title("Actual vs Predicted Meropenem Values (E. coli) - Test Dataset")
plt.legend()
plt.show()




# Print all test predictions
for i in range(len(y_test_scaled)):
    print(f"Actual: {y_test_scaled[i]:.2f}, Predicted: {pred_scaled[i]:.2f}")
    # Given actual and predicted values for a test set


x_labels = [f"Sample {i+1}" for i in range(len(actual_values))]  # Labels for x-axis
x = np.arange(len(actual_values))  # x positions

plt.figure(figsize=(10, 5))

# Bar width
bar_width = 0.35

# Plot bars for actual and predicted values
plt.bar(x - bar_width/2, actual_values, bar_width, label="Actual", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, predicted_values, bar_width, label="Predicted", color="orange", alpha=0.7)

plt.xlabel("Samples")
plt.ylabel("Meropenem Value")
plt.title("Actual vs Predicted Meropenem Values (E. coli)")

# X-axis labels and positioning
plt.xticks(ticks=x, labels=x_labels)

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(actual_values, marker="o", linestyle="-", color="blue", label="Actual")
plt.plot(predicted_values, marker="x", linestyle="--", color="orange", label="Predicted")

plt.xlabel("Sample Index")
plt.ylabel("Meropenem Value")
plt.title("Trend Comparison: Actual vs Predicted Meropenem Values")
plt.xticks(range(len(actual_values)), labels=x_labels, rotation=45)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

